In [1]:
import numpy as np
import pandas as pd
import random
import csv
import geopandas as gpd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, MaxAbsScaler
from sklearn.impute import KNNImputer
import sys
import pickle

sys.path.insert(0, "../../../Modules")

pd.set_option('display.max_columns', None)
random.seed(0)
np.random.seed(0)

from utils import *
from utils import process_greek

enc = 'utf-8'
shapefiles_folder = "C:/Users/dimit/Documents/noa hoard/Greece Shapefiles"

c:\Users\dimit\AppData\Local\Programs\Python\Python39\lib\site-packages\geopandas\_compat.py:112: UserWarning: The Shapely GEOS version (3.10.3-CAPI-1.16.1) is incompatible with the GEOS version PyGEOS was compiled with (3.10.4-CAPI-1.16.2). Conversions between both will be slow.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
NUTS0 = 'GR'
NUTS2 = 'Thessaly'

In [4]:
YEAR = 2023
MONTH = 'June'
PERIOD = '1st'

In [5]:
model = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Linear_model_new.pkl', 'rb'))
scaler = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Scaler_new.pkl', 'rb'))
imputer = pickle.load(open(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Imputer_new.pkl', 'rb'))

In [6]:
data_test = read_data(f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_WNV_Dataset_{YEAR}-{MONTH}-{PERIOD}.csv')
data_test.head()

,x,y,dt_placement,nuts2,lau1,day,month,week,year,day_sin,day_cos,month_sin,month_cos,week_sin,week_cos,area,population,population_density,eq_distance,ndvi,ndmi,ndwi,ndbi,ndvi_mean,ndmi_mean,ndwi_mean,ndbi_mean,ndvi_std,ndmi_std,ndwi_std,ndbi_std,lst,lst_day,lst_night,lst_jan_day_mean,lst_jan_night_mean,lst_feb_day_mean,lst_feb_night_mean,lst_mar_day_mean,lst_mar_night_mean,lst_apr_day_mean,lst_apr_night_mean,acc_rainfall_1week,acc_rainfall_2week,acc_rainfall_jan,distance_to_coast,distance_to_river,slope_mean_1km,aspect_mean_200m,elevation_mean_1km,hillshade_mean_1km,fs_area_1km,flow_accu_200m,lc_prop1,lc_prop1_assessment,lc_prop2,lc_prop2_assessment,lc_prop3,lc_prop3_assessment,lc_type1,lc_type2,lc_type3,lc_type4,lc_type5,lw,qc,mosq_pred,mosq_previous,case
0,22.76105,39.69186,2023-06-01,θεσσαλιας,αγιας,1,6,22,2023,0.201299,0.97953,1.224647e-16,-1.0,0.508531,-0.861044,661.8,10705.0,17.3,45.75488,0.619638,0.295823,-0.499309,-0.295823,0.592041,0.288053,-0.476920,-0.288053,0.042702,0.040975,0.033709,0.040975,19.398513,24.878750,13.918276,10.418186,4.056923,10.285211,3.423839,16.670680,6.227661,17.857476,7.916795,0.479727,14.596283,127.173282,8282.127022,1331.619869,1,123.585589,121.767859,183.036108,0.0,21.311709,31,90,30,90.0,30,90,10,10,1,6,6,2,0,21,0,0
1,22.72671,39.12560,2023-06-01,θεσσαλιας,αλμυρου,1,6,22,2023,0.201299,0.97953,1.224647e-16,-1.0,0.508531,-0.861044,905.4,16004.0,20.6,45.24728,0.470555,0.228169,-0.381034,-0.228169,0.408219,0.245974,-0.314974,-0.245974,0.052834,0.035967,0.046198,0.035967,20.326875,27.520417,13.133333,11.219310,3.845804,10.731281,3.006374,16.701534,6.377584,19.115966,8.094541,1.810689,13.447538,232.999498,10017.207897,1610.666116,14,259.585198,342.571837,171.163634,0.0,3.415750,11,80,10,72.0,10,72,1,1,7,1,1,2,0,18,0,0
2,23.99842,39.24585,2023-06-01,θεσσαλιας,αλοννησου,1,6,22,2023,0.201299,0.97953,1.224647e-16,-1.0,0.508531,-0.861044,129.6,3153.0,21.2,46.00175,-0.173575,0.246094,0.349643,-0.246094,-0.168335,0.236517,0.341655,-0.236517,0.004380,0.009322,0.005392,0.009322,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.443456,0.461510,253.654365,1484.683259,55302.082778,24,302.637680,165.397106,195.027810,0.0,1.676994,21,94,20,94.0,20,94,8,8,4,1,1,2,0,1,0,0
3,21.48510,39.29630,2023-06-01,θεσσαλιας,αργιθεας,1,6,22,2023,0.201299,0.97953,1.224647e-16,-1.0,0.508531,-0.861044,372.9,3515.0,9.3,44.78626,0.517589,0.217258,-0.414658,-0.217258,0.468445,0.217090,-0.367562,-0.217090,0.089583,0.063360,0.071952,0.063360,14.670769,19.836154,9.505385,6.942786,1.831897,7.062325,-0.605892,13.120876,3.695211,13.925591,4.285066,0.742939,9.178791,193.742062,19327.278832,1925.318354,27,284.268894,1127.254453,234.678996,0.0,1.719393,21,88,20,88.0,20,88,8,8,4,1,1,2,0,22,0,0
4,22.93502,39.38117,2023-06-01,θεσσαλιας,βολου,1,6,22,2023,0.201299,0.97953,1.224647e-16,-1.0,0.508531,-0.861044,385.6,138865.0,374.6,45.57293,0.454709,0.158620,-0.381657,-0.158620,0.405253,0.162856,-0.332698,-0.162856,0.044248,0.039037,0.036832,0.039037,20.368485,27.083333,13.653636,12.159420,4.198851,11.394699,3.845437,17.241419,5.994943,19.899661,8.124530,1.105031,2.301892,179.345895,5935.142744,3071.988689,5,143.181190,234.931813,174.426021,0.0,4.643743,31,91,30,91.0,30,91,10,10,1,6,6,2,0,33,0,0


In [7]:
features_to_remove = ['x', 'y', 'eq_distance','day', 'month', 'week', 'year', 'lc_prop1_assessment',
                    'lc_prop2', 'lc_prop2_assessment', 'lc_prop3', 'lc_prop3_assessment',
                    'lc_type2', 'lc_type3', 'lc_type4', 'lc_type5', 'lw', 'qc','ndvi_mean', 'ndmi_mean', 'ndwi_mean', 'ndbi_mean', 'ndvi_std',
                    'ndmi_std', 'ndwi_std', 'ndbi_std',]

In [8]:
object_cols = data_test.select_dtypes(include=['object']).columns.to_list()
removed_cols = features_to_remove

X_test = data_test.drop(columns = ['case'] + object_cols + removed_cols)
y_test = data_test['case']

X_test = scaler.transform(X_test)
X_test = imputer.fit_transform(X_test)

results_test = inference_lin_model(model, data_test, X_test, y_test)

In [9]:
results_test.drop(columns='case', inplace= True)
results_test

,x,y,lau1,day,month,year,score
0,22.34107,39.61048,λαρισαιων,1,6,2023,0.312306
1,22.54288,39.85090,τεμπων,1,6,2023,0.072912
2,22.29611,39.76297,τυρναβου,1,6,2023,0.072627
3,21.48510,39.29630,αργιθεας,1,6,2023,0.068727
4,22.93502,39.38117,βολου,1,6,2023,0.048284
5,22.15952,39.95174,ελασσονας,1,6,2023,0.042953
6,22.02629,39.60300,φαρκαδονας,1,6,2023,0.035395
7,22.09265,39.48079,παλαμα,1,6,2023,0.032614
8,22.08995,39.24048,σοφαδων,1,6,2023,0.031233
9,22.43796,39.30328,φαρσαλων,1,6,2023,0.031050


In [10]:
bins_path = f'../../data/{NUTS2}/{NUTS0}_{NUTS2}_Bins_{YEAR}.csv'

bins = []

with open(bins_path, mode='r', newline='') as file:
    reader = csv.reader(file)
    for row in reader:
        bins.extend(map(float, row))

print(bins)

[0.0, 0.0080309448997414, 0.0860446074909823, 0.5109550767195234, 0.8201371322355278, 0.9305335064919854, 1.0]


In [11]:
results_test['risk_class'] = pd.cut(results_test['score'], bins=bins, labels=False, include_lowest=True)
results_test

,x,y,lau1,day,month,year,score,risk_class
0,22.34107,39.61048,λαρισαιων,1,6,2023,0.312306,2
1,22.54288,39.85090,τεμπων,1,6,2023,0.072912,1
2,22.29611,39.76297,τυρναβου,1,6,2023,0.072627,1
3,21.48510,39.29630,αργιθεας,1,6,2023,0.068727,1
4,22.93502,39.38117,βολου,1,6,2023,0.048284,1
5,22.15952,39.95174,ελασσονας,1,6,2023,0.042953,1
6,22.02629,39.60300,φαρκαδονας,1,6,2023,0.035395,1
7,22.09265,39.48079,παλαμα,1,6,2023,0.032614,1
8,22.08995,39.24048,σοφαδων,1,6,2023,0.031233,1
9,22.43796,39.30328,φαρσαλων,1,6,2023,0.031050,1


In [12]:
# results_test.to_csv(f"../../data/{NUTS2}/results/{NUTS0}_{NUTS2}_Results_{YEAR}-{MONTH}-{PERIOD}.csv", encoding = enc, index = False)

In [13]:
##TODO Visualisation of results